# COVID-19 Behavior Analysis
## 1. Data Preprocessing
### 1.1 Load and Inspect the Dataset

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("COVID-19BehaviorData_CAN2022.csv")

# Check the dataset dimensions
print("Dataset shape:", df.shape)

# Display the first few rows
df.head()

Dataset shape: (6430, 512)


/tmp/ipykernel_926/988661424.py:2: DtypeWarning: Columns (4,505,506,507,508,509,510,511) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("COVID-19BehaviorData_CAN2022.csv")


,RecordNo,endtime,qweek,i1_health,i2_health,i7a_health,i3_health,i4_health,i5_health_1,i5_health_2,...,vac_man_96,vac_man_99,q_other,had_covid,vac_boost_beyond,future_1,future_2,had_covid_2,long_covid,future_3
0,41589,06/01/2022 18:02,week 44,,0,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41731,03/01/2022 23:48,week 44,,1,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41756,10/01/2022 04:53,week 44,,0,,,,,,...,Yes,No,"Oui bar , restaurent",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41846,01/01/2022 13:15,week 44,,5,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,41847,01/01/2022 19:12,week 44,,150,,,,,,...,No,No,__NA__,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Count special missing value codes
(df == "__NA__").sum().sort_values(ascending=False).head(20)

vac12_other            6348
q_other                6190
vac12_booster_other    6044
i14_health_other       5929
Soc2_open              4290
i12_health_29             0
Soc1_1                    0
Soc1_2                    0
Soc1_3                    0
Soc2_1                    0
Soc2_2                    0
i12_health_28             0
Soc2_4                    0
Soc2_5                    0
Soc2_6                    0
vac_1                     0
vac2_1                    0
Soc2_3                    0
i12_health_27             0
work8                     0
dtype: int64

In [4]:
# Count values that contain only whitespace
whitespace_counts = df.select_dtypes(include="object").apply(
    lambda col: col.str.fullmatch(r"\s*").sum()
)

whitespace_counts.sort_values(ascending=False).head(20)

WAH6                 6430
V1_1                 6430
V1_2                 6430
V1_3                 6430
V1_98                6430
V3_me                6430
V3_me_other          6430
V3_baby              6430
WAH7_1               6430
WAH7_2               6430
WAH7_3               6430
WAH7_4               6430
WAH7_5               6430
WAH7_6               6430
WAH7_7               6430
WAH7_99              6430
Vent_2               6430
WAH4                 6430
WAH5                 6430
child_education_1    6430
dtype: int64

In [5]:
print("Columns with whitespace values:",
      (whitespace_counts > 0).sum())

print("Total whitespace values:",
      whitespace_counts.sum())

Columns with whitespace values: 449
Total whitespace values: 2652523


### 1.2 Data Cleaning

In [6]:
# Create a copy for preprocessing
df_clean = df.copy()

# Convert whitespace-only values to NaN
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)

# Convert special missing value codes to NaN
df_clean = df_clean.replace('__NA__', np.nan)

# Check missing values after cleaning
print("Total missing values:", df_clean.isna().sum().sum())

print("Columns with all missing values:",
      df_clean.isna().all().sum())

Total missing values: 2698511
Columns with all missing values: 353


/tmp/ipykernel_926/62661265.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)


In [7]:
# Remove columns with 100% missing values
df_clean = df_clean.dropna(axis=1, how="all")

# Check updated dataset dimensions
print("Dataset shape after removing empty columns:", df_clean.shape)

Dataset shape after removing empty columns: (6430, 159)


In [8]:
# Examine target variable
df_clean["vac7"].value_counts(dropna=False)

vac7
Moderately    2388
Very much     2213
A little      1075
Not at all     754
Name: count, dtype: int64

### 1.2 Define the Target Variable

In [9]:
# Create binary target variable
trust_mapping = {
    "Very much": 0,
    "Moderately": 0,
    "A little": 1,
    "Not at all": 1
}

df_clean["low_trust"] = df_clean["vac7"].map(trust_mapping)

# Check target distribution
print(df_clean["low_trust"].value_counts())
print(df_clean["low_trust"].value_counts(normalize=True) * 100)

low_trust
0    4601
1    1829
Name: count, dtype: int64
low_trust
0    71.55521
1    28.44479
Name: proportion, dtype: float64


/tmp/ipykernel_926/2281991887.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean["low_trust"] = df_clean["vac7"].map(trust_mapping)


### 1.3 Feature Selection

In [10]:
# Define candidate predictors
candidate_features = [
    "age", "gender", "region", "employment_status",
    "household_size", "household_children",
    "r1_1", "r1_2",
    "i12_health_1", "i12_health_2",
    "i12_health_3", "i12_health_5",
    "had_covid", "had_covid_2",
    "WCRex1", "WCRex2"
]

# Check missing percentages and data types
feature_summary = pd.DataFrame({
    "Data Type": df_clean[candidate_features].dtypes,
    "Missing (%)": df_clean[candidate_features].isna().mean() * 100,
    "Unique Values": df_clean[candidate_features].nunique()
})

feature_summary.round(2)

,Data Type,Missing (%),Unique Values
age,int64,0.00,73
gender,object,0.00,2
region,object,0.00,12
employment_status,object,0.00,7
household_size,object,0.00,10
household_children,object,0.00,8
r1_1,object,0.00,7
r1_2,object,0.00,7
i12_health_1,object,0.00,5
i12_health_2,object,15.57,5


In [11]:
# Inspect unique values of selected predictors
for col in [
    "household_size",
    "household_children",
    "r1_1",
    "r1_2",
    "i12_health_2",
    "WCRex1",
    "WCRex2"
]:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False))


household_size:
household_size
2                    2294
1                    1386
3                     940
4                     881
5                     380
Prefer not to say     204
6                     158
7                      70
8 or more              60
Don't know             57
Name: count, dtype: int64

household_children:
household_children
0                    4292
1                     823
2                     688
3                     247
4                     166
5 or more             108
Prefer not to say      97
Don't know              9
Name: count, dtype: int64

r1_1:
r1_1
7 - Agree       1306
4               1195
5                904
1 – Disagree     876
3                827
6                675
2                647
Name: count, dtype: int64

r1_2:
r1_2
4               1530
1 – Disagree    1063
5                914
3                884
2                771
7 - Agree        750
6                518
Name: count, dtype: int64

i12_health_2:
i12_health_2
Always    

In [12]:
selected_features = [
    "age", "gender", "region", "employment_status",
    "household_size", "household_children",
    "r1_1", "r1_2",
    "i12_health_1", "i12_health_2",
    "i12_health_3", "i12_health_5",
    "WCRex1", "WCRex2"
]

# Check special responses in selected features
for col in selected_features:
    special = df_clean[col].astype("string").str.contains(
        "don't know|prefer not|refused",
        case=False,
        na=False
    )
    if special.any():
        print(col, ":", special.sum())

household_size : 261
household_children : 106
WCRex1 : 301
WCRex2 : 327


In [13]:
# Select predictors
X = df_clean[selected_features].copy()

# Rename columns for readability
X = X.rename(columns={
    "household_children": "number_of_children",
    "r1_1": "perceived_covid_severity",
    "r1_2": "perceived_infection_risk",
    "i12_health_1": "mask_wearing_frequency",
    "i12_health_2": "handwashing_frequency",
    "i12_health_3": "hand_sanitizer_frequency",
    "i12_health_5": "avoiding_infected_people",
    "WCRex1": "government_covid_response",
    "WCRex2": "healthcare_system_confidence"
})

# Check updated column names
print(X.columns.tolist())

['age', 'gender', 'region', 'employment_status', 'household_size', 'number_of_children', 'perceived_covid_severity', 'perceived_infection_risk', 'mask_wearing_frequency', 'handwashing_frequency', 'hand_sanitizer_frequency', 'avoiding_infected_people', 'government_covid_response', 'healthcare_system_confidence']


### 1.4 Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split

# Define target variable
y = df_clean["low_trust"]

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check dataset dimensions
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Check target distribution
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (5144, 14)
Test set: (1286, 14)

Training target distribution:
low_trust
0    0.715591
1    0.284409
Name: proportion, dtype: float64

Test target distribution:
low_trust
0    0.715397
1    0.284603
Name: proportion, dtype: float64


In [15]:
# Group features by data type
numerical_features = [
    "age",
    "household_size",
    "number_of_children"
]

ordinal_features = [
    "perceived_covid_severity",
    "perceived_infection_risk",
    "mask_wearing_frequency",
    "handwashing_frequency",
    "hand_sanitizer_frequency",
    "avoiding_infected_people"
]

categorical_features = [
    "gender",
    "region",
    "employment_status",
    "government_covid_response",
    "healthcare_system_confidence"
]

# Check that all features are included exactly once
all_features = numerical_features + ordinal_features + categorical_features

print("Total features:", len(all_features))
print("All features accounted for:", set(all_features) == set(X_train.columns))
print("No duplicate features:", len(all_features) == len(set(all_features)))

Total features: 14
All features accounted for: True
No duplicate features: True


### 1.5 Numerical Feature Cleaning

In [16]:
# Define mappings for top-coded values
household_mapping = {
    "8 or more": "8",
    "5 or more": "5"
}

# Clean household variables
for col in ["household_size", "number_of_children"]:
    X_train[col] = pd.to_numeric(
        X_train[col].replace(household_mapping),
        errors="coerce"
    )

    X_test[col] = pd.to_numeric(
        X_test[col].replace(household_mapping),
        errors="coerce"
    )

# Check results
print(X_train[numerical_features].dtypes)

print("\nMissing values in training set:")
print(X_train[numerical_features].isna().sum())

age                     int64
household_size        float64
number_of_children    float64
dtype: object

Missing values in training set:
age                     0
household_size        200
number_of_children     77
dtype: int64


In [17]:
# Convert risk perception variables using the original data
for old_col, new_col in [
    ("r1_1", "perceived_covid_severity"),
    ("r1_2", "perceived_infection_risk")
]:
    X_train[new_col] = pd.to_numeric(
        df_clean.loc[X_train.index, old_col].str.extract(r"^(\d+)")[0],
        errors="coerce"
    )

    X_test[new_col] = pd.to_numeric(
        df_clean.loc[X_test.index, old_col].str.extract(r"^(\d+)")[0],
        errors="coerce"
    )

# Check missing values
print(X_train[
    ["perceived_covid_severity", "perceived_infection_risk"]
].isna().sum())

perceived_covid_severity    0
perceived_infection_risk    0
dtype: int64


In [18]:
# Define frequency mapping
frequency_mapping = {
    "Not at all": 1,
    "Rarely": 2,
    "Sometimes": 3,
    "Frequently": 4,
    "Always": 5
}

# Map original responses to numerical values
behavior_columns = {
    "i12_health_1": "mask_wearing_frequency",
    "i12_health_2": "handwashing_frequency",
    "i12_health_3": "hand_sanitizer_frequency",
    "i12_health_5": "avoiding_infected_people"
}

for original_col, new_col in behavior_columns.items():
    X_train[new_col] = df_clean.loc[X_train.index, original_col].map(frequency_mapping)
    X_test[new_col] = df_clean.loc[X_test.index, original_col].map(frequency_mapping)

# Check results
for col in behavior_columns.values():
    print(f"\n{col}")
    print(X_train[col].value_counts(dropna=False).sort_index())


mask_wearing_frequency
mask_wearing_frequency
1     281
2     118
3     214
4     527
5    4004
Name: count, dtype: int64

handwashing_frequency
handwashing_frequency
1.0      59
2.0      53
3.0     214
4.0     921
5.0    3087
NaN     810
Name: count, dtype: int64

hand_sanitizer_frequency
hand_sanitizer_frequency
1     229
2     250
3     554
4    1251
5    2860
Name: count, dtype: int64

avoiding_infected_people
avoiding_infected_people
1     309
2     118
3     366
4     606
5    3745
Name: count, dtype: int64


### 1.6 Preprocessing Pipeline

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Ordinal preprocessing
ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


### 1.7 Validate Preprocessed Data

In [20]:
# Check unique values and missing values in ordinal features
for col in ordinal_features:
    print(f"\n{col}")
    print("Unique values:", sorted(X_train[col].dropna().unique()))
    print("Missing values:", X_train[col].isna().sum())


perceived_covid_severity
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Missing values: 0

perceived_infection_risk
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Missing values: 0

mask_wearing_frequency
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Missing values: 0

handwashing_frequency
Unique values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
Missing values: 810

hand_sanitizer_frequency
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Missing values: 0

avoiding_infected_people
Unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Missing values: 0


In [21]:
# Fit preprocessing pipeline using corrected training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the same transformations to test data
X_test_processed = preprocessor.transform(X_test)

# Convert to numeric arrays for validation
X_train_processed = np.asarray(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    dtype=float
)

X_test_processed = np.asarray(
    X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed,
    dtype=float
)

# Validate processed datasets
print("Training shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

print("Training missing values:", np.isnan(X_train_processed).sum())
print("Test missing values:", np.isnan(X_test_processed).sum())

print("Training infinite values:", np.isinf(X_train_processed).sum())
print("Test infinite values:", np.isinf(X_test_processed).sum())

Training shape: (5144, 40)
Test shape: (1286, 40)
Training missing values: 0
Test missing values: 0
Training infinite values: 0
Test infinite values: 0


In [22]:
# Get feature names after preprocessing
processed_feature_names = preprocessor.get_feature_names_out()

print("Total processed features:", len(processed_feature_names))

Total processed features: 40


### 1.8 Save Training and Test Data

In [23]:
from pathlib import Path

# Create output directory
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save training and test datasets
X_train.to_csv(output_dir / "X_train.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

print("Training and test datasets saved successfully!")

Training and test datasets saved successfully!


### 1.9 Preprocessing Summary

The original dataset contains 6,430 observations. The following preprocessing steps were completed:

- **Data cleaning:** Replaced blank strings and `__NA__` with missing values and removed columns containing only missing values.
- **Target variable:** Created `low_trust` from `vac7`, where 1 indicates low vaccine trust and 0 indicates higher vaccine trust.
- **Feature selection:** Selected 14 predictors covering demographics, COVID-19 risk perceptions, preventive behaviors, and public health attitudes.
- **Train-test split:** Split the data into 80% training and 20% testing sets using stratified sampling.
- **Feature encoding:** Converted numerical variables to numeric format and encoded ordinal variables using 1–7 and 1–5 scales.
- **Preprocessing pipeline:** Created a pipeline for missing value imputation, standard scaling, and one-hot encoding.
- **Validation:** Tested the pipeline on the training and test sets, producing 40 features with no missing or infinite values.

The pipeline was fitted on the training data only and applied to the test data. The saved CSV files contain the 14 selected features after basic cleaning and encoding, without the final pipeline transformations. The preprocessing pipeline should be included during model training and cross-validation to avoid data leakage.